In [28]:
# some imports 
import numpy as np 
import networkx as nx
import copy 
from itertools import product
from collections import Counter
import matplotlib.pyplot as plt
from numpy import random
#D-Wave Ocean SDK related
import dimod 
from dwave.system import DWaveSampler
from dwave.samplers import SimulatedAnnealingSampler
# from dwave.cloud import Client
# client = Client.from_config()
from dwave.embedding import embed_ising, unembed_sampleset
from minorminer import find_embedding
import pickle
from pathlib import Path

seed = 1224
random.seed(seed)

max_square = 47 #48*48 or 2304
max_er = 174
max_regular = 896
max_full = 171
max_ba1 = 3412
max_ba2 = 980

qpu_sampler = DWaveSampler(solver="Advantage_system4")

In [29]:
embeddings = pickle.loads(Path("../pkls/embeddings.pkl").read_bytes())
SA_sample_sets = pickle.loads(Path("../pkls/SA_sample_sets.pkl").read_bytes())
QA_sample_sets = pickle.loads(Path("../pkls/QA_sample_sets.pkl").read_bytes())
sa_results = pickle.loads(Path("../pkls/sa_results.pkl").read_bytes())
qa_results = pickle.loads(Path("../pkls/qa_results.pkl").read_bytes())
graphs = pickle.loads(Path("../pkls/graphs.pkl").read_bytes())
labels = pickle.loads(Path("../pkls/labels.pkl").read_bytes())

In [30]:
def assign_uniform_weights(graph):
    for u, v in graph.edges():
        graph[u][v]["weight"] = random.uniform(-1,1)
    for node in graph.nodes():
        graph.nodes[node]["weight"] = random.uniform(-1,1)
    return graph

def assign_random_weights(graph):
    seq = [-1,1]
    for u, v in graph.edges():
        graph[u][v]["weight"] = random.choice(seq)
    for node in graph.nodes():
        graph.nodes[node]["weight"] = random.choice(seq)
    return graph

def assign_gauss_weights(graph):
    for u, v in graph.edges():
        graph[u][v]["weight"] = random.normal(loc=0.0, scale=1.0)
    for node in graph.nodes():
        graph.nodes[node]["weight"] = random.normal(loc=0.0, scale=1.0)
    return graph

def create_regular():
    return nx.random_regular_graph(d=3, n=max_regular, seed=seed)
def create_square():
    return nx.grid_2d_graph(n=max_square, m=max_square)
def create_er():
    return nx.erdos_renyi_graph(n=max_er, p=0.5, seed=seed)
def create_full():
    return nx.complete_graph(n=max_full)
def create_ba1():
    return nx.barabasi_albert_graph(n=max_ba1, m=1, seed=seed)
def create_ba2():
    return nx.barabasi_albert_graph(n=max_ba2, m=2, seed=seed)

In [31]:
graph_templates = [create_regular(), create_er(), create_full(), create_square(), create_ba1(), create_ba2()]
weight_methods = [assign_uniform_weights, assign_gauss_weights, assign_random_weights]

base_labels = ["Uniform Regular","Normal Regular","Binary Regular",
          "Uniform Erdos-Renyi","Normal Erdos-Renyi","Binary Erdos-Renyi",
          "Uniform Full","Normal Full","Binary Full",
          "Uniform Square","Normal Square","Binary Square",
          "Uniform Barabasi-Albert (1)","Normal Barabasi-Albert (1)","Binary Barabasi-Albert (1)",
          "Uniform Barabasi-Albert","Normal Barabasi-Albert","Binary Barabasi-Albert"]

graphs = []
labels = []

for g, g_template in enumerate(graph_templates):
    for w, w_method in enumerate(weight_methods):
        for i in range(0,10):
            graphs.append(w_method(copy.deepcopy(g_template)))
            labels.append(base_labels[g * len(weight_methods) + w]+f" {i}.")

In [ ]:
def run_QA(graph, label, embed):
    num_reads = 2000
    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")

    h_embedded, J_embedded = embed_ising(h, J, embed, qpu_sampler.adjacency)
    
    # QA_sampleset = qpu_sampler.sample_ising(h_embedded, J_embedded, num_reads=num_reads)
    # bqm = dimod.BinaryQuadraticModel(h, J, offset=0.0, vartype=dimod.SPIN)
    # unembedded_QA_sampleset = unembed_sampleset(QA_sampleset, embed, bqm)

    energies = unembedded_QA_sampleset.data_vectors["energy"]
    expected_energy = np.mean(energies)
    print('Simulated Annealing for: ', label)
    print(f'{num_reads} samples were drawn using Simulated Annealing sampler')
    print('number of variables: \t', graph.number_of_nodes())
    print('number of samples: \t', len(unembedded_QA_sampleset.record))
    print('best sample: \t\t', unembedded_QA_sampleset.first.sample)
    print('best energy: \t\t', unembedded_QA_sampleset.first.energy)
    print('expected energy: \t', expected_energy)
    return (unembedded_QA_sampleset)

from dwave.samplers import SteepestDescentSolver

SD_solver = SteepestDescentSolver()

def run_SD(graph, SA_sampleset, label):
    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")

    SD_sampleset = SD_solver.sample_ising(
        h, J,
        initial_states=SA_sampleset,
        num_reads=len(SA_sampleset),
    )

    SA_energies = SA_sampleset.data_vectors["energy"]
    SD_energies = SD_sampleset.data_vectors["energy"]
    return SD_sampleset, {
        "label": label,
        "SA_expected": np.mean(SA_energies),
        "SA_best": SA_energies.min(),
        "SD_expected": np.mean(SD_energies),
        "SD_best": SD_energies.min(),
    }


In [33]:
print(labels)

['Uniform Regular 0.', 'Uniform Regular 1.', 'Uniform Regular 2.', 'Uniform Regular 3.', 'Uniform Regular 4.', 'Uniform Regular 5.', 'Uniform Regular 6.', 'Uniform Regular 7.', 'Uniform Regular 8.', 'Uniform Regular 9.', 'Normal Regular 0.', 'Normal Regular 1.', 'Normal Regular 2.', 'Normal Regular 3.', 'Normal Regular 4.', 'Normal Regular 5.', 'Normal Regular 6.', 'Normal Regular 7.', 'Normal Regular 8.', 'Normal Regular 9.', 'Binary Regular 0.', 'Binary Regular 1.', 'Binary Regular 2.', 'Binary Regular 3.', 'Binary Regular 4.', 'Binary Regular 5.', 'Binary Regular 6.', 'Binary Regular 7.', 'Binary Regular 8.', 'Binary Regular 9.', 'Uniform Erdos-Renyi 0.', 'Uniform Erdos-Renyi 1.', 'Uniform Erdos-Renyi 2.', 'Uniform Erdos-Renyi 3.', 'Uniform Erdos-Renyi 4.', 'Uniform Erdos-Renyi 5.', 'Uniform Erdos-Renyi 6.', 'Uniform Erdos-Renyi 7.', 'Uniform Erdos-Renyi 8.', 'Uniform Erdos-Renyi 9.', 'Normal Erdos-Renyi 0.', 'Normal Erdos-Renyi 1.', 'Normal Erdos-Renyi 2.', 'Normal Erdos-Renyi 3.',

In [ ]:
copies_per_combo = 10

# pt4_results = []
# for i, graph in enumerate(graphs):
#     embedding_itr = i // copies_per_combo
#     pt4_results.append(run_QA(graph, labels[i], embeddings[embedding_itr]))

with open('pt4_results.pkl', 'wb') as file:
    pickle.dump(pt4_results, file)

Simulated Annealing for:  Uniform Regular 0.
2000 samples were drawn using Simulated Annealing sampler
number of variables: 	 896
number of samples: 	 2000
best sample: 		 {0: np.int8(1), 1: np.int8(1), 2: np.int8(1), 3: np.int8(1), 4: np.int8(-1), 5: np.int8(1), 6: np.int8(1), 7: np.int8(1), 8: np.int8(-1), 9: np.int8(-1), 10: np.int8(1), 11: np.int8(1), 12: np.int8(1), 13: np.int8(-1), 14: np.int8(-1), 15: np.int8(-1), 16: np.int8(-1), 17: np.int8(-1), 18: np.int8(-1), 19: np.int8(-1), 20: np.int8(-1), 21: np.int8(1), 22: np.int8(-1), 23: np.int8(1), 24: np.int8(1), 25: np.int8(1), 26: np.int8(1), 27: np.int8(1), 28: np.int8(-1), 29: np.int8(-1), 30: np.int8(-1), 31: np.int8(-1), 32: np.int8(1), 33: np.int8(-1), 34: np.int8(1), 35: np.int8(-1), 36: np.int8(-1), 37: np.int8(-1), 38: np.int8(1), 39: np.int8(-1), 40: np.int8(1), 41: np.int8(-1), 42: np.int8(1), 43: np.int8(-1), 44: np.int8(1), 45: np.int8(1), 46: np.int8(1), 47: np.int8(1), 48: np.int8(1), 49: np.int8(-1), 50: np.int8(1

In [36]:
print(len(pt4_results))

180


In [35]:

SD_sample_sets = []
SD_summaries = []
for graph, SA_ss, label in zip(graphs, SA_sample_sets, labels):
    ss, summary = run_SD(graph, SA_ss, label)
    SD_sample_sets.append(ss)
    SD_summaries.append(summary)

ValueError: mismatch between variables in 'initial_states' and 'bqm'